# M1-B2 — Classifieur de motifs d'insatisfaction

Une section par tâche du brief. Les cellules `TODO` sont à compléter.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"   # avant tout « import keras » : Keras utilisera PyTorch

from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
CACHE = DATA / "cache"
CACHE.mkdir(exist_ok=True)
MODELS = ROOT / "models"

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

## Tâche 1 — Démarrer

Cette cellule télécharge le modèle d'embeddings au premier lancement (environ 470 Mo) 

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer(MODEL_NAME)
vecteurs = encoder.encode(["Le lit était très inconfortable.", "Le matelas était défoncé."])
vecteurs.shape   # (2, 384) : deux phrases, 384 nombres chacune

## Tâche 2 — Explorer le corpus

In [ ]:
avis = pd.read_csv(DATA / "avis_motifs.csv")
recents = pd.read_csv(DATA / "avis_recents.csv")
avis.head()

In [ ]:
# TODO : répartition des motifs, longueur des avis, lecture d'une vingtaine d'avis avec leur étiquette

### Notre avis sur les étiquettes du LLM

*Note ici les étiquettes qui vous semblent douteuses.*

## Tâche 3 — Une référence : TF-IDF + régression logistique

In [ ]:
# TODO : découpage entraînement / test stratifié, pipeline TF-IDF + LogisticRegression, F1 macro

## Tâche 4 — Des embeddings

Encode les avis **une seule fois**, puis sauvegarde les vecteurs (par exemple `np.save(CACHE / "avis.npy", ...)`)
pour ne pas les recalculer à chaque essai.

In [ ]:
# TODO : encoder avis et récents (une fois), sauvegarder, régression logistique sur les vecteurs

## Tâche 5 — Un petit réseau Keras

In [ ]:
#Note : les données préparées E_train et train proviennent des cellules précédentes

import keras
from sklearn.preprocessing import LabelEncoder


keras.utils.set_random_seed(42)
le = LabelEncoder().fit(avis["motif"])
y_train = le.transform(train["motif"])

reseau = keras.Sequential([
    keras.Input(shape=(E_train.shape[1],)),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(len(le.classes_), activation="softmax"),
])
reseau.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])


historique = reseau.fit(
    E_train, y_train, validation_split=0.2, epochs=100, batch_size=32, verbose=0,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
)
print("arrêt après", len(historique.history["loss"]), "époques")



## Tâche 6 — Comparer et analyser

In [ ]:
# TODO : F1 macro sur le test ET sur les avis récents, matrices de confusion, temps de prédiction, taille

### 10 erreurs examinées

| avis_id | étiquette | prédiction | erreur du modèle ou étiquette douteuse ? |
|---|---|---|---|
| | | | |

In [ ]:
# TODO : sauvegarder le modèle retenu dans MODELS